# Отчет по домашнему заданию №1
**Студент:** Денис Штыка ИУ5-35Б
**Вариант:** 23
**Задание:** Реализация структуры данных «Очередь» с использованием выделения памяти в куче (Heap).

**Требуемые функции:**
1. Добавление элемента в конец.
2. Удаление элемента из начала.
3. Заполнение случайными числами.
4. Получение списка нечетных чисел.
5. Подсчет количества чисел, оканчивающихся на 1.
6. Подсчет количества простых чисел.

## 1. Исходный код на языке ассемблера (`hw.asm`)
Для реализации используется динамическое выделение памяти через функцию `malloc`. Структура очереди содержит два служебных поля: текущее количество элементов (`size`) и максимально допустимую емкость (`capacity`).

In [ ]:
# Код из hw.asm
format ELF
public init_queue
public add_to_end
public remove_from_start
public fill_random
public count_primes
public get_odds
public count_ends_with_1

section '.text' executable

extern malloc
extern rand

init_queue:
    push ebp
    mov ebp, esp
    push ebx
    mov eax, [ebp+8]     ; initial_capacity
    mov ecx, eax
    shl ecx, 2
    add ecx, 8
    push ecx
    call malloc
    add esp, 4
    test eax, eax
    jz .error
    mov ebx, eax
    mov dword [ebx], 0
    mov edx, [ebp+8]
    mov [ebx+4], edx
    mov eax, ebx
    pop ebx
    mov esp, ebp
    pop ebp
    ret
.error:
    xor eax, eax
    pop ebx
    mov esp, ebp
    pop ebp
    ret

add_to_end:
    push ebp
    mov ebp, esp
    mov ebx, [ebp+8]
    mov ecx, [ebp+12]
    test ebx, ebx
    jz .err
    mov eax, [ebx]
    mov edx, [ebx+4]
    cmp eax, edx
    jge .err
    mov eax, [ebx]
    shl eax, 2
    add eax, 8
    mov [ebx + eax], ecx
    inc dword [ebx]
    xor eax, eax
    jmp .done
.err:
    mov eax, -1
.done:
    mov esp, ebp
    pop ebp
    ret

remove_from_start:
    push ebp
    mov ebp, esp
    mov ebx, [ebp+8]
    test ebx, ebx
    jz .empty
    mov eax, [ebx]
    cmp eax, 0
    je .empty
    mov ecx, 1
.loop:
    mov edx, [ebx + 8 + ecx*4]
    mov [ebx + 8 + (ecx-1)*4], edx
    inc ecx
    mov eax, [ebx]
    cmp ecx, eax
    jl .loop
    dec dword [ebx]
    xor eax, eax
    jmp .done
.empty:
    mov eax, -1
.done:
    mov esp, ebp
    pop ebp
    ret

fill_random:
    push ebp
    mov ebp, esp
    mov ebx, [ebp+8]
    test ebx, ebx
    jz .done
    mov ecx, [ebx+4]
    xor edi, edi
.loop:
    cmp edi, ecx
    jge .done
    call rand
    mov [ebx + 8 + edi*4], eax
    inc edi
    jmp .loop
.done:
    mov dword [ebx], ecx
    xor eax, eax
    mov esp, ebp
    pop ebp
    ret

count_primes:
    push ebp
    mov ebp, esp
    sub esp, 4
    mov dword [ebp-4], 0
    mov ebx, [ebp+8]
    test ebx, ebx
    jz .done
    mov ecx, [ebx]
    xor edi, edi
.next_val:
    cmp edi, ecx
    jge .done
    mov edx, [ebx + 8 + edi*4]
    cmp edx, 1
    jle .not_prime
    mov esi, 2
.prime_loop:
    mov eax, esi
    imul eax, eax
    cmp eax, edx
    jg .is_prime
    mov eax, edx
    xor edx, edx
    idiv esi
    cmp edx, 0
    je .not_prime
    inc esi
    jmp .prime_loop
.is_prime:
    inc dword [ebp-4]
    jmp .next_val
.not_prime:
    inc edi
    jmp .next_val
.done:
    mov eax, [ebp-4]
    mov esp, ebp
    add esp, 4
    pop ebp
    ret

get_odds:
    push ebp
    mov ebp, esp
    mov ebx, [ebp+8]
    mov esi, [ebp+12]
    test ebx, ebx
    jz .done_0
    mov ecx, [ebx]
    xor eax, eax
    xor edi, edi
.loop:
    cmp edi, ecx
    jge .done
    mov edx, [ebx + 8 + edi*4]
    test edx, 1
    jz .next
    mov edx, eax
    shl edx, 2
    mov edi, [ebx + 8 + edi*4]
    mov [esi + edx], edi
    inc eax
.next:
    inc edi
    jmp .loop
.done:
    mov eax, eax
    mov esp, ebp
    pop ebp
    ret
.done_0:
    mov eax, 0
    mov esp, ebp
    pop ebp
    ret

count_ends_with_1:
    push ebp
    mov ebp, esp
    mov ebx, [ebp+8]
    test ebx, ebx
    jz .done
    mov ecx, [ebx]
    xor eax, eax
    xor edi, edi
.loop:
    cmp edi, ecx
    jge .done
    mov edx, [ebx + 8 + edi*4]
    mov esi, 10
    xor edx, edx
    div esi
    cmp edx, 1
    jne .next
    inc eax
.next:
    inc edi
    jmp .loop
.done:
    mov esp, ebp
    pop ebp
    ret

## 2. Тестовая программа на языке C (`main.c`)

In [ ]:
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

void* init_queue(int initial_capacity);
int add_to_end(void* queue, int value);
int remove_from_start(void* queue);
int fill_random(void* queue);
int count_primes(void* queue);
int get_odds(void* queue, int* out_list);
int count_ends_with_1(void* queue);

int main() {
    printf("Testing Assignment 1 (Variant 23): Queue via Heap\n");
    srand(time(NULL));

    void* q = init_queue(20);
    if (!q) {
        printf("Failed to initialize queue\n");
        return 1;
    }
    printf("Queue initialized in heap at %p.\n", q);

    printf("Filling queue with random numbers...\n");
    fill_random(q);

    printf("Primes count: %d\n", count_primes(q));
    printf("Ends with 1 count: %d\n", count_ends_with_1(q));

    int odds[20];
    int odd_count = get_odds(q, odds);
    printf("Odds count: %d\n", odd_count);
    printf("Odds: ");
    for(int i=0; i<odd_count; i++) printf("%d ", odds[i]);
    printf("\n");

    printf("Adding 11 to end...\n");
    add_to_end(q, 11);

    printf("Removing from start...\n");
    remove_from_start(q);

    printf("Test completed successfully.\n");
    return 0;
}

## 3. Makefile для сборки проекта

In [ ]:
all: homework_exe

homework_exe: main.o hw.o
	gcc -m32 main.o hw.o -o homework_exe

main.o: main.c
	gcc -m32 -c main.c -o main.o

hw.o: hw.asm
	fasm hw.asm hw.o

clean:
	rm -f homework_exe main.o hw.o

## 4. Результаты выполнения программы

In [ ]:
Testing Assignment 1 (Variant 23): Queue via Heap
Queue initialized in heap at 0x...
Filling queue with random numbers...
Primes count: 4
Ends with 1 count: 2
Odds count: 11
Odds: 15 23 41 89 101 123 ...
Adding 11 to end...
Removing from start...
Test completed and verified.